# 05 — Locked final evaluation, realistic gap tests, and flagged reconstruction

Only run final evaluation after the declared models/protocol have been frozen. This notebook verifies data, model and code hashes and protects existing outputs. Artificial-gap analysis is a secondary test on observed targets; no reference truth is created for naturally missing values. Field validation remains a separate study component.

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/CHT_SM_v2")
else:
    WORK = ROOT / "work_v2"
WORK.mkdir(parents=True, exist_ok=True)
print("Persistent work folder:", WORK)


In [ ]:
from chtsm.data import read_config, load_exports, audit_frame, save_json
CONFIG_PATH = WORK / "study_config.json"
config = read_config(CONFIG_PATH)
# Move the monthly FULL exports from the Drive export folder into this folder.
# Pilot exports have a different prefix and must not be mixed into the full table.
EXPORT_DIR = WORK / "exports"
CSV_PATHS = sorted(EXPORT_DIR.glob("chtsm_v2_full_*.csv"))
if not CSV_PATHS:
    raise FileNotFoundError("Put completed monthly chtsm_v2_full_*.csv exports in WORK/exports first.")
frame = load_exports(CSV_PATHS, config)


In [ ]:
from chtsm.models import evaluate_final
RUN = WORK / "runs" / "study_v2_temporal_001"
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    raise RuntimeError("Final evaluation is disabled. Freeze the run and explicitly set RUN_FINAL_TEST=True when ready.")
metrics, reconstruction = evaluate_final(frame, RUN, allow_final_test=True)
import pandas as pd
summary = pd.DataFrame([{"model": name, "rmse": value["micro"]["rmse"],
                        "macro_cell_rmse": value["macro_cell_rmse"], "r2": value["micro"]["r2"],
                        "interval_coverage": value["empirical_interval_coverage"]}
                       for name, value in metrics.items()])
display(summary)
summary.to_csv(RUN / "final_test" / "summary.csv", index=False)

In [ ]:
from chtsm.gaps import run_gap_experiments
gap_results = run_gap_experiments(frame, RUN, lengths=(1,3,7,14), seeds=(42,123,2025))
gap_table = pd.DataFrame([{"gap_days": r["requested_gap_days"], "seed": r["seed"],
                         "model": name, "rmse": m["rmse"], "scored": r["common_scored_targets"]}
                        for r in gap_results for name, m in r["models"].items()])
display(gap_table.groupby(["gap_days", "model"]).rmse.agg(["mean","std"]))
gap_table.to_csv(RUN / "final_test" / "gap_summary.csv", index=False)

Persistence and linear interpolation receive the same retained satellite observations; hidden targets are removed before those baselines are computed. Interpolation can use future retained values because this is retrospective reconstruction. Learned models use only exogenous predictors. All methods are scored on the same recoverable reference coverage. Report this information-set distinction and the count of excluded edge cases.

In [ ]:
import matplotlib.pyplot as plt
# Cell-level output avoids treating changing-coverage regional means as observations.
CELL_ID = reconstruction.cell_id.iloc[0]  # Choose a cell from your actual grid.
r = reconstruction[reconstruction.cell_id == CELL_ID]
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(r.date, r.complete_sm, lw=1, color="steelblue", label="Selected-model completed product")
known = r.target_sm.notna(); ax.scatter(r.loc[known,"date"],r.loc[known,"target_sm"],s=7,color="black",label="QA-passing SMAP")
missing = r.is_reconstructed
ax.scatter(r.loc[missing,"date"],r.loc[missing,"complete_sm"],s=7,color="tomato",label="Reconstructed; reference unknown")
ax.set(title=f"SMAP product reconstruction — {CELL_ID}; not independent field validation", ylabel="m³/m³")
ax.legend();fig.tight_layout();fig.savefig(RUN / "final_test" / "example_cell.png", dpi=150);plt.show()
print("Unsupported cells:", reconstruction.loc[reconstruction.unsupported_cell,"cell_id"].nunique())
print("Range-extrapolation flags:", int(reconstruction.outside_training_ranges.sum()))
print("Files:", RUN / "final_test")

Before manuscript claims: examine per-cell, season, forest-fraction and slope scores, paired block-bootstrap sensor comparisons, multi-seed variability, interval coverage/width, and the effect of poor sensor availability. Forest fraction is not canopy density; current masks/QA do not diagnose RFI. Prediction ranges and support flags are diagnostics, not full out-of-distribution detection. Predictions are withheld for cells with too few training targets. Calibration-domain transfer and natural missing-condition accuracy remain unverified.